<a href="https://colab.research.google.com/github/vallem21-03/mi-proyecto/blob/main/streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 2.2 – Análisis de datos de streaming con pandas

In [1]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv('streaming.csv')
print(df.shape)
df.head()

(6214, 16)


,User_ID,Session_ID,Device_ID,Video_ID,Duration_Watched (minutes),Genre,Country,Age,Gender,Subscription_Status,Ratings,Languages,Device_Type,Location,Playback_Quality,Interaction_Events
0,eb4f9229-74df-45f6-baac-cf19241b8b30,cb2142a7-0750-49ed-b8ac-a975fe1ff69a,232,11,90.044525,Sci-Fi,Sudan,56,Female,Premium,3,Spanish,Smartphone,Reedshire,4K,73
1,661d4b59-4328-410a-901c-1e3b4c40c334,3bc0a662-b353-4015-8b0c-55ceb510d13a,549,85,68.973479,Comedy,Cuba,52,Male,Premium,3,Chinese,Desktop,Stevenhaven,SD,22
2,dd3fe9e9-ea82-4891-ab93-8a47c80e3251,bd545b4a-9f54-4e87-b9f8-15ae20b44f22,844,50,42.511343,Comedy,Japan,14,Female,Premium,3,Spanish,Tablet,Vaughntown,HD,41
3,a1b3365b-1d00-4ddf-bc43-02fc9c10c680,0441086d-c59e-478d-a496-5c5b995ecfdb,201,38,53.316660,Documentary,Lao People's Democratic Republic,36,Male,Premium,2,Spanish,Laptop,East Raymondbury,SD,40
4,338d3f91-5f1c-4590-8803-324901826406,0295f01d-7f15-4799-856c-90c688697ef8,700,31,69.437786,Action,Bangladesh,31,Male,Premium,4,German,Laptop,Michaelchester,HD,41


## 1. Eliminar columnas (id de sesión, id de dispositivo, lenguaje, localización, calidad de playback e interacciones)

In [2]:
cols_eliminar = ['Session_ID', 'Device_ID', 'Languages', 'Location',
                 'Playback_Quality', 'Interaction_Events']
df = df.drop(columns=cols_eliminar)

# Comprobación de calidad
print(df.shape)                       # mismas filas, 6 columnas menos (16 -> 10)
print(df.columns.tolist())            # quedan solo las que se usan
print(df.dtypes)                      # edad, rating y minutos vistos son numéricos
print(df['Gender'].unique())          # categorías exactas: 'Female' / 'Male'
print(df['Subscription_Status'].unique())
print(df.isna().sum().sum(), 'nulos')

(6214, 10)
['User_ID', 'Video_ID', 'Duration_Watched (minutes)', 'Genre', 'Country', 'Age', 'Gender', 'Subscription_Status', 'Ratings', 'Device_Type']
User_ID                        object
Video_ID                        int64
Duration_Watched (minutes)    float64
Genre                          object
Country                        object
Age                             int64
Gender                         object
Subscription_Status            object
Ratings                         int64
Device_Type                    object
dtype: object
['Female' 'Male']
['Premium' 'Free']
0 nulos


**Validación:** 6.214 filas se mantienen y pasamos de 16 a 10 columnas. No hay nulos, `Age`, `Ratings` y `Duration_Watched (minutes)` son numéricas y `Gender` solo contiene `Female` y `Male`, así que los filtros posteriores funcionan.

## 2. Género preferido por los hombres

In [6]:
hombres = df[df['Gender'] == 'Male']

# Preferencia = género más visto (número de sesiones)
print(hombres['Genre'].value_counts())

print("-------")

# Comprobamos con otras dos métricas
print(hombres.groupby('Genre')['Duration_Watched (minutes)'].sum().round(0).sort_values(ascending=False))

print("-------")

print(hombres.groupby('Genre')['Ratings'].mean().round(3).sort_values(ascending=False))

Genre
Documentary    552
Action         517
Sci-Fi         513
Drama          502
Comedy         491
Thriller       491
Name: count, dtype: int64
-------
Genre
Documentary    34751.0
Action         31506.0
Drama          30945.0
Sci-Fi         29565.0
Thriller       29453.0
Comedy         28322.0
Name: Duration_Watched (minutes), dtype: float64
-------
Genre
Action         3.068
Drama          3.020
Documentary    3.004
Thriller       2.992
Comedy         2.921
Sci-Fi         2.862
Name: Ratings, dtype: float64


**Resultado:** el género más visto por los hombres es **Documentary** (552 sesiones y 34.751 minutos). Sin embargo, el mejor valorado por ellos es Action (3,07), por lo que 'preferido' depende de si se mide por consumo o por valoración.

## 3. ¿Los smartphones son el dispositivo donde más tiempo se mira streaming?

In [ ]:
tiempo_dispositivo = (df.groupby('Device_Type')['Duration_Watched (minutes)']
                        .agg(['sum', 'mean', 'count'])
                        .round(2)
                        .sort_values('sum', ascending=False))
tiempo_dispositivo

**Resultado:** **no**. Por tiempo total, Smartphone queda en 3.er lugar (75.226 min) y Smart TV es el primero (78.276 min). Por tiempo medio por sesión, Smartphone y Laptop están arriba (≈61 min), pero la diferencia con el resto es de apenas 1-2 minutos.

## 4. Edades que más pagan Premium

In [7]:
premium = df[df['Subscription_Status'] == 'Premium']

# Edades con más suscriptores Premium (valores absolutos)
print(premium['Age'].value_counts().head(10))

# Tasa de Premium por franja de edad
franjas = pd.cut(df['Age'], [0, 17, 25, 35, 45, 55, 65, 100])
(df.groupby(franjas, observed=True)['Subscription_Status']
   .apply(lambda s: (s == 'Premium').mean()).round(3))

Age
40    72
13    69
52    66
57    64
50    63
16    60
45    59
20    59
51    58
27    57
Name: count, dtype: int64


,Subscription_Status
Age,
"(0, 17]",0.519
"(17, 25]",0.490
"(25, 35]",0.475
"(35, 45]",0.487
"(45, 55]",0.516
"(55, 65]",0.510
"(65, 100]",0.502


**Resultado:** las edades con más usuarios Premium son **40 años (72)**, 13 (69) y 52 (66). Por franjas, la proporción de Premium es parecida en todas (47 %-52 %); las de 0-17 y 45-55 años son las más altas.

## 5. Vídeo con más tiempo de reproducción

In [9]:
videos = (df.groupby('Video_ID')['Duration_Watched (minutes)']
            .agg(['sum', 'mean', 'count'])
            .round(2)
            .sort_values('sum', ascending=False))
videos.head(5)

,sum,mean,count
Video_ID,,,
11,5088.84,66.09,77
44,4726.56,63.87,74
59,4584.42,60.32,76
73,4541.80,68.82,66
24,4505.75,61.72,73


**Resultado:** el **vídeo 11** acumula más tiempo de reproducción (5.088,8 min en 77 sesiones). Cabe destacar que el vídeo 73, con menos sesiones, tiene la mayor media por sesión (68,8 min).

## 6. Género que más gustó a las mujeres (rating)

In [8]:
mujeres = df[df['Gender'] == 'Female']
(mujeres.groupby('Genre')['Ratings']
        .agg(['mean', 'count'])
        .round(3)
        .sort_values('mean', ascending=False))

,mean,count
Genre,,
Comedy,3.098,542
Sci-Fi,3.061,528
Action,3.016,513
Drama,3.012,493
Thriller,2.943,559
Documentary,2.848,513


**Resultado:** el género mejor valorado por las mujeres es **Comedy** (3,10 de media sobre 5, con 542 valoraciones). Documentary es el peor valorado (2,85).

## Conclusiones

**Hechos observados en los datos**
- El dataset tiene 6.214 sesiones, sin nulos y con usuarios únicos. Hombres y mujeres están equilibrados (3.066 y 3.148).
- Los hombres ven sobre todo documentales; las mujeres valoran mejor la comedia, y los documentales son el género que peor puntúan. Es un contraste llamativo: el género más visto por ellos es el peor valorado por ellas.
- Los smartphones **no** son el dispositivo con más tiempo total; lo es la Smart TV. Los tiempos por sesión son casi iguales en los cinco dispositivos (59-61 min).
- Cerca de la mitad de los usuarios es Premium (3.103 de 6.214) y la proporción apenas cambia con la edad.
- El vídeo 11 es el que más tiempo acumula.

**Hipótesis (los datos no bastan para afirmarlas)**
- Las diferencias son pequeñas (por ejemplo, 3,10 frente a 3,06 de rating en las mujeres, o 72 frente a 69 Premium por edad). Pueden deberse al azar, y se necesitarían pruebas estadísticas para confirmarlas.
- Que Smart TV lidere en tiempo total puede deberse a que tiene más sesiones (1.306), no a que se vea más tiempo en cada una. Habría que comprobarlo con más datos de uso.
- Un vídeo con más tiempo acumulado no tiene por qué ser mejor: puede haber sido visto en más sesiones. Hay que mirar también el tiempo medio y el rating.
- Se podría pensar que las edades concretas con más Premium (40, 13, 52) reflejan un patrón real, pero con ~50 usuarios por edad es más prudente hablar de franjas amplias, donde no hay diferencias claras.
- Estos datos no permiten explicar por qué se prefiere un género o un dispositivo, solo describir qué ocurre.